# Ensemble Learning Methods: Random Forest, Boosting, Bagging, and Stacking

This notebook covers ensemble learning techniques including Bagging, Random Forest, Boosting (AdaBoost, Gradient Boosting), and Stacking implemented using PyTorch and scikit-learn.

## 1. Import Required Libraries

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                              AdaBoostClassifier, GradientBoostingClassifier,
                              StackingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch Version: {torch.__version__}')

## 2. Ensemble Learning Theory

### Key Concepts

**Ensemble Learning**: Combining multiple models to improve prediction performance

**Types:**
1. **Bagging** (Bootstrap Aggregating): Parallel ensemble, reduces variance
2. **Boosting**: Sequential ensemble, reduces bias
3. **Stacking**: Meta-learning approach combining diverse models

### Why Ensembles Work
- **Wisdom of crowds**: Multiple models make better decisions
- **Error reduction**: Averaging reduces variance
- **Bias-variance tradeoff**: Different methods address different aspects

## 3. Load and Prepare Dataset

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names

print(f'Dataset: Breast Cancer Wisconsin')
print(f'Samples: {X.shape[0]}')
print(f'Features: {X.shape[1]}')
print(f'Classes: {target_names}')

# Create DataFrame
df = pd.DataFrame(X, columns=feature_names)
df['target'] = y
print('\nClass distribution:')
print(df['target'].value_counts())
df.head()

In [ ]:
# Split and scale data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## 4. Baseline Model - Single Decision Tree

In [ ]:
# Train single decision tree
dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train, y_train)
y_pred_baseline = dt_baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f'Baseline Decision Tree Accuracy: {baseline_accuracy:.4f}')

# Cross-validation
cv_scores_baseline = cross_val_score(dt_baseline, X_train, y_train, cv=5)
print(f'Cross-validation accuracy: {cv_scores_baseline.mean():.4f} (+/- {cv_scores_baseline.std():.4f})')

## 5. Bagging (Bootstrap Aggregating)

In [ ]:
# Train Bagging Classifier
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=50,
    max_samples=0.8,
    max_features=0.8,
    bootstrap=True,
    random_state=42
)

bagging.fit(X_train, y_train)
y_pred_bagging = bagging.predict(X_test)
bagging_accuracy = accuracy_score(y_test, y_pred_bagging)

print(f'Bagging Accuracy: {bagging_accuracy:.4f}')
print(f'Improvement over baseline: {(bagging_accuracy - baseline_accuracy)*100:.2f}%')

# Cross-validation
cv_scores_bagging = cross_val_score(bagging, X_train, y_train, cv=5)
print(f'Cross-validation accuracy: {cv_scores_bagging.mean():.4f} (+/- {cv_scores_bagging.std():.4f})')

## 6. Random Forest

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print(f'Random Forest Accuracy: {rf_accuracy:.4f}')
print(f'Improvement over baseline: {(rf_accuracy - baseline_accuracy)*100:.2f}%')

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(feature_importance.head(10)['feature'], feature_importance.head(10)['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Effect of number of trees
n_estimators_range = [1, 5, 10, 20, 50, 100, 150, 200]
rf_accuracies = []

for n in n_estimators_range:
    rf_temp = RandomForestClassifier(n_estimators=n, random_state=42)
    rf_temp.fit(X_train, y_train)
    y_pred_temp = rf_temp.predict(X_test)
    rf_accuracies.append(accuracy_score(y_test, y_pred_temp))

plt.figure(figsize=(10, 6))
plt.plot(n_estimators_range, rf_accuracies, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Trees', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Random Forest: Accuracy vs Number of Trees', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. AdaBoost (Adaptive Boosting)

In [ ]:
# Train AdaBoost
adaboost = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=3),
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)

adaboost.fit(X_train, y_train)
y_pred_ada = adaboost.predict(X_test)
ada_accuracy = accuracy_score(y_test, y_pred_ada)

print(f'AdaBoost Accuracy: {ada_accuracy:.4f}')
print(f'Improvement over baseline: {(ada_accuracy - baseline_accuracy)*100:.2f}%')

# Cross-validation
cv_scores_ada = cross_val_score(adaboost, X_train, y_train, cv=5)
print(f'Cross-validation accuracy: {cv_scores_ada.mean():.4f} (+/- {cv_scores_ada.std():.4f})')

In [ ]:
# Effect of learning rate
learning_rates = [0.01, 0.1, 0.5, 1.0, 1.5, 2.0]
ada_lr_accuracies = []

for lr in learning_rates:
    ada_temp = AdaBoostClassifier(n_estimators=50, learning_rate=lr, random_state=42)
    ada_temp.fit(X_train, y_train)
    y_pred_temp = ada_temp.predict(X_test)
    ada_lr_accuracies.append(accuracy_score(y_test, y_pred_temp))

plt.figure(figsize=(10, 6))
plt.plot(learning_rates, ada_lr_accuracies, marker='o', linewidth=2, markersize=8)
plt.xlabel('Learning Rate', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('AdaBoost: Accuracy vs Learning Rate', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Gradient Boosting

In [ ]:
# Train Gradient Boosting
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
gb_accuracy = accuracy_score(y_test, y_pred_gb)

print(f'Gradient Boosting Accuracy: {gb_accuracy:.4f}')
print(f'Improvement over baseline: {(gb_accuracy - baseline_accuracy)*100:.2f}%')

# Feature importance
feature_importance_gb = pd.DataFrame({
    'feature': feature_names,
    'importance': gb.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(feature_importance_gb.head(10)['feature'], feature_importance_gb.head(10)['importance'])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importances - Gradient Boosting')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Voting Classifier (Hard and Soft Voting)

In [ ]:
# Create base estimators
estimators = [
    ('lr', LogisticRegression(random_state=42, max_iter=1000)),
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('svc', SVC(probability=True, random_state=42))
]

# Hard Voting
voting_hard = VotingClassifier(estimators=estimators, voting='hard')
voting_hard.fit(X_train_scaled, y_train)
y_pred_voting_hard = voting_hard.predict(X_test_scaled)
voting_hard_accuracy = accuracy_score(y_test, y_pred_voting_hard)

# Soft Voting
voting_soft = VotingClassifier(estimators=estimators, voting='soft')
voting_soft.fit(X_train_scaled, y_train)
y_pred_voting_soft = voting_soft.predict(X_test_scaled)
voting_soft_accuracy = accuracy_score(y_test, y_pred_voting_soft)

print(f'Hard Voting Accuracy: {voting_hard_accuracy:.4f}')
print(f'Soft Voting Accuracy: {voting_soft_accuracy:.4f}')

## 10. Stacking Classifier

In [ ]:
# Create base learners
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]

# Meta-learner
meta_learner = LogisticRegression(random_state=42, max_iter=1000)

# Create stacking classifier
stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)

stacking.fit(X_train_scaled, y_train)
y_pred_stacking = stacking.predict(X_test_scaled)
stacking_accuracy = accuracy_score(y_test, y_pred_stacking)

print(f'Stacking Classifier Accuracy: {stacking_accuracy:.4f}')
print(f'Improvement over baseline: {(stacking_accuracy - baseline_accuracy)*100:.2f}%')

# Individual base learner performance
print('\nBase Learner Performance:')
for name, model in base_learners:
    model.fit(X_train_scaled, y_train)
    y_pred_base = model.predict(X_test_scaled)
    base_acc = accuracy_score(y_test, y_pred_base)
    print(f'{name}: {base_acc:.4f}')

## 11. Comprehensive Comparison

In [ ]:
# Compare all ensemble methods
results = {
    'Decision Tree\n(Baseline)': baseline_accuracy,
    'Bagging': bagging_accuracy,
    'Random Forest': rf_accuracy,
    'AdaBoost': ada_accuracy,
    'Gradient\nBoosting': gb_accuracy,
    'Hard Voting': voting_hard_accuracy,
    'Soft Voting': voting_soft_accuracy,
    'Stacking': stacking_accuracy
}

plt.figure(figsize=(14, 7))
colors = ['gray', 'skyblue', 'lightgreen', 'lightcoral', 'gold', 'plum', 'lavender', 'lightblue']
bars = plt.bar(results.keys(), results.values(), color=colors, edgecolor='black', linewidth=1.5)
plt.xlabel('Method', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Comprehensive Comparison of Ensemble Methods', fontsize=15, fontweight='bold')
plt.ylim([0.9, 1.0])
plt.xticks(rotation=45, ha='right')

for i, (method, acc) in enumerate(results.items()):
    plt.text(i, acc + 0.003, f'{acc:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print sorted results
print('\nRanked Performance:')
sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
for rank, (method, acc) in enumerate(sorted_results, 1):
    print(f'{rank}. {method.replace(chr(10), " ")}: {acc:.4f}')

## 12. Confusion Matrices Comparison

In [ ]:
# Create confusion matrices for key methods
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

predictions = [
    ('Baseline DT', y_pred_baseline),
    ('Random Forest', y_pred_rf),
    ('AdaBoost', y_pred_ada),
    ('Gradient Boosting', y_pred_gb),
    ('Soft Voting', y_pred_voting_soft),
    ('Stacking', y_pred_stacking)
]

for idx, (name, y_pred) in enumerate(predictions):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=target_names, yticklabels=target_names)
    axes[idx].set_title(name, fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## 13. ROC Curves Comparison

In [ ]:
# Plot ROC curves
plt.figure(figsize=(12, 8))

models = [
    ('Random Forest', rf),
    ('Gradient Boosting', gb),
    ('AdaBoost', adaboost),
    ('Stacking', stacking)
]

for name, model in models:
    if name == 'Stacking':
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        y_proba = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Ensemble Methods Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 14. Key Takeaways

### Ensemble Methods Summary

#### Bagging (Bootstrap Aggregating)
- **Strategy**: Train multiple models on random subsets with replacement
- **Reduces**: Variance
- **Example**: Random Forest
- **Best for**: High variance models (deep decision trees)
- **Parallelizable**: Yes

#### Random Forest
- **Enhancement**: Bagging + random feature selection
- **Advantages**: Handles high-dimensional data, robust to outliers
- **Feature importance**: Built-in capability

#### Boosting
- **Strategy**: Sequential training, focus on misclassified samples
- **Reduces**: Bias
- **Variants**: AdaBoost, Gradient Boosting, XGBoost
- **Best for**: Weak learners (shallow trees)
- **Parallelizable**: Limited

#### AdaBoost
- **Mechanism**: Adjusts sample weights based on errors
- **Sensitive to**: Outliers and noise

#### Gradient Boosting
- **Mechanism**: Fits new models to residual errors
- **Often**: Best performance but slower training

#### Voting
- **Hard voting**: Majority class prediction
- **Soft voting**: Average predicted probabilities
- **Requires**: Diverse base models

#### Stacking
- **Strategy**: Meta-learner combines base model predictions
- **Complexity**: Highest
- **Performance**: Often best when properly tuned

### When to Use What
- **High variance problem**: Use Bagging/Random Forest
- **High bias problem**: Use Boosting
- **Need interpretability**: Random Forest (feature importance)
- **Maximum performance**: Gradient Boosting or Stacking
- **Limited computational resources**: Random Forest or Bagging